# Run 322 artifact probe

Draft notebook for `run_id=322b4485-557c-434b-b2e1-0776d15a515b`.

This notebook loads the pinned BTCUSDT `1h` price artifacts and the two signal matrices used by the run:

- `ma.dema` from `signals/1h/ma.dema/signals.i8.npy`
- `ma.ema` from `signals/1h/ma.ema/signals.i8.npy`

Pinned runtime identity:

- slot: `slot_a`
- generation: `1`
- asof_date: `2026-04-02`
- manifest_hash: `13df35a144a9706b6b40c949f71ddc5dd23d60da10bbda81c12e26f9676faaae`
- timeframe: `1h`
- symbol: `BTCUSDT`


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import yaml

RUN_ID = "322b4485-557c-434b-b2e1-0776d15a515b"
TIMEFRAME = "1h"
RUN_TIME_RANGE_START = datetime(2017, 10, 6, 22, 55, tzinfo=timezone.utc)
RUN_TIME_RANGE_END = datetime(2026, 3, 31, 22, 55, tzinfo=timezone.utc)
RUN_TIME_RANGE_START_MS = int(RUN_TIME_RANGE_START.timestamp() * 1000)
RUN_TIME_RANGE_END_MS = int(RUN_TIME_RANGE_END.timestamp() * 1000)

ARTIFACT_ROOT = Path("/opt/roehub/state/backtest_artifacts/v2/binance/spot/BTCUSDT/slot_a")
PRICE_DIR = ARTIFACT_ROOT / "prices" / TIMEFRAME
DEMA_DIR = ARTIFACT_ROOT / "signals" / TIMEFRAME / "ma.dema"
EMA_DIR = ARTIFACT_ROOT / "signals" / TIMEFRAME / "ma.ema"

SLOT_MANIFEST_PATH = ARTIFACT_ROOT / "manifest.yaml"
PRICE_OPEN_TIME_PATH = PRICE_DIR / "open_time.i64.npy"
PRICE_CLOSE_TIME_PATH = PRICE_DIR / "close_time.i64.npy"
PRICE_OHLCV_PATH = PRICE_DIR / "ohlcv.f32.npy"
DEMA_MANIFEST_PATH = DEMA_DIR / "manifest.yaml"
EMA_MANIFEST_PATH = EMA_DIR / "manifest.yaml"
DEMA_SIGNAL_PATH = DEMA_DIR / "signals.i8.npy"
EMA_SIGNAL_PATH = EMA_DIR / "signals.i8.npy"

REQUEST_INDICATOR_GRIDS = [
    {
        "indicator_id": "ma.dema",
        "sources": ["close"],
        "window_range": [5, 200],
    },
    {
        "indicator_id": "ma.ema",
        "sources": ["high", "ohlc4"],
        "window_range": [5, 200],
    },
]

for path in [
    SLOT_MANIFEST_PATH,
    PRICE_OPEN_TIME_PATH,
    PRICE_CLOSE_TIME_PATH,
    PRICE_OHLCV_PATH,
    DEMA_MANIFEST_PATH,
    EMA_MANIFEST_PATH,
    DEMA_SIGNAL_PATH,
    EMA_SIGNAL_PATH,
]:
    print(f"{path}: exists={path.exists()}")


In [ ]:
slot_manifest = yaml.safe_load(SLOT_MANIFEST_PATH.read_text())
dema_manifest = yaml.safe_load(DEMA_MANIFEST_PATH.read_text())
ema_manifest = yaml.safe_load(EMA_MANIFEST_PATH.read_text())

prices_by_timeframe = {item["timeframe"]: item for item in slot_manifest["prices"]}
price_manifest = prices_by_timeframe[TIMEFRAME]

print("slot:", slot_manifest["slot"], "generation:", slot_manifest["slot_generation"], "asof_date:", slot_manifest["asof_date"])
print("price bars:", price_manifest["coverage"]["bar_count"])
print("dema signals shape:", tuple(dema_manifest["signals"]["shape"]))
print("ema signals shape:", tuple(ema_manifest["signals"]["shape"]))
print("request indicator grids:", REQUEST_INDICATOR_GRIDS)


In [ ]:
price_open_time = np.load(PRICE_OPEN_TIME_PATH, mmap_mode="r")
price_close_time = np.load(PRICE_CLOSE_TIME_PATH, mmap_mode="r")
price_ohlcv = np.load(PRICE_OHLCV_PATH, mmap_mode="r")
dema_signals = np.load(DEMA_SIGNAL_PATH, mmap_mode="r")
ema_signals = np.load(EMA_SIGNAL_PATH, mmap_mode="r")

print("price_open_time:", price_open_time.shape, price_open_time.dtype)
print("price_close_time:", price_close_time.shape, price_close_time.dtype)
print("price_ohlcv:", price_ohlcv.shape, price_ohlcv.dtype)
print("dema_signals:", dema_signals.shape, dema_signals.dtype)
print("ema_signals:", ema_signals.shape, ema_signals.dtype)


In [ ]:
open_time_index = price_open_time.astype("datetime64[ms]")
close_time_index = price_close_time.astype("datetime64[ms]")
time_mask = (price_open_time >= RUN_TIME_RANGE_START_MS) & (price_open_time <= RUN_TIME_RANGE_END_MS)

print("bars inside run time range:", int(time_mask.sum()))
print("first matching bar:", np.datetime_as_string(open_time_index[time_mask][0], timezone="UTC"))
print("last matching bar:", np.datetime_as_string(open_time_index[time_mask][-1], timezone="UTC"))


In [ ]:
sample_indices = np.flatnonzero(time_mask)[:5]
sample_prices = [
    {
        "open_time": np.datetime_as_string(open_time_index[idx], timezone="UTC"),
        "close_time": np.datetime_as_string(close_time_index[idx], timezone="UTC"),
        "open": float(price_ohlcv[idx, 0]),
        "high": float(price_ohlcv[idx, 1]),
        "low": float(price_ohlcv[idx, 2]),
        "close": float(price_ohlcv[idx, 3]),
        "volume": float(price_ohlcv[idx, 4]),
    }
    for idx in sample_indices
]
sample_prices


In [ ]:
signal_probe = [
    {
        "open_time": np.datetime_as_string(open_time_index[idx], timezone="UTC"),
        "dema_row_0": int(dema_signals[0, idx]),
        "ema_row_0": int(ema_signals[0, idx]),
    }
    for idx in range(12)
]
signal_probe


## Notes

- The two `signals.i8.npy` files above are the exact artifact-backed matrices used by the run.
- Each file stores the full indicator matrix for one indicator family on the pinned `1h` timeline.
- The run-specific subset is narrower than the full matrix:
  - `ma.dema`: `source=close`, `window=5..200`
  - `ma.ema`: `source in {high, ohlc4}`, `window=5..200`
- This draft notebook intentionally stops at loading the exact pinned `npy` artifacts and probing timeline alignment.
- Next extension: reconstruct the row mapping for the selected sources/windows from the v2 signal rules/defaults catalog and then slice the relevant rows from each matrix.


## Additional 15m artifact loads for BTCUSDT spot

This section adds a broader `15m` probe on the same pinned slot for three extra indicators from different families:

- `ma.sma`
- `momentum.roc`
- `volatility.stddev`

It also loads:

- `prices/15m/*`
- derived price calculations (`open`, `high`, `low`, `close`, `hlc3`, `ohlc4`)
- strict `hit_times/1m/*` TP/SL tables and levels

The existing `1h` run-specific experiment above is kept intact.


In [ ]:
TIMEFRAME_15M = "15m"
PRICE_DIR_15M = ARTIFACT_ROOT / "prices" / TIMEFRAME_15M
HIT_TIMES_DIR_1M = ARTIFACT_ROOT / "hit_times" / "1m"

EXTRA_INDICATORS_15M = {
    "ma.sma": {
        "family": "ma",
        "sources": ["close", "hlc3", "ohlc4", "low", "high", "open"],
        "param_name": "window",
        "param_values": list(range(5, 201)),
    },
    "momentum.roc": {
        "family": "momentum",
        "sources": ["close", "hlc3", "ohlc4", "low", "high", "open"],
        "param_name": "window",
        "param_values": [5, 7, 10, 14, 21, 28, 42, 63, 84, 126],
    },
    "volatility.stddev": {
        "family": "volatility",
        "sources": ["close", "hlc3", "ohlc4", "low", "high", "open"],
        "param_name": "window",
        "param_values": [10, 14, 20, 28, 42, 56, 84, 126],
        "signal_defaults": {"long_delta_periods": -5, "short_delta_periods": -10},
    },
}

PRICE_OPEN_TIME_15M_PATH = PRICE_DIR_15M / "open_time.i64.npy"
PRICE_CLOSE_TIME_15M_PATH = PRICE_DIR_15M / "close_time.i64.npy"
PRICE_OHLCV_15M_PATH = PRICE_DIR_15M / "ohlcv.f32.npy"
HIT_TIMES_MANIFEST_PATH = HIT_TIMES_DIR_1M / "manifest.yaml"
TP_VALUES_PATH = HIT_TIMES_DIR_1M / "tp_values.f32.npy"
SL_VALUES_PATH = HIT_TIMES_DIR_1M / "sl_values.f32.npy"
LONG_TP_PATH = HIT_TIMES_DIR_1M / "long_tp.u32.npy"
SHORT_TP_PATH = HIT_TIMES_DIR_1M / "short_tp.u32.npy"
LONG_SL_PATH = HIT_TIMES_DIR_1M / "long_sl.u32.npy"
SHORT_SL_PATH = HIT_TIMES_DIR_1M / "short_sl.u32.npy"

EXTRA_SIGNAL_PATHS_15M = {
    indicator_id: {
        "manifest": ARTIFACT_ROOT / "signals" / TIMEFRAME_15M / indicator_id / "manifest.yaml",
        "signals": ARTIFACT_ROOT / "signals" / TIMEFRAME_15M / indicator_id / "signals.i8.npy",
    }
    for indicator_id in EXTRA_INDICATORS_15M
}

paths_to_check = [
    PRICE_OPEN_TIME_15M_PATH,
    PRICE_CLOSE_TIME_15M_PATH,
    PRICE_OHLCV_15M_PATH,
    HIT_TIMES_MANIFEST_PATH,
    TP_VALUES_PATH,
    SL_VALUES_PATH,
    LONG_TP_PATH,
    SHORT_TP_PATH,
    LONG_SL_PATH,
    SHORT_SL_PATH,
] + [value for item in EXTRA_SIGNAL_PATHS_15M.values() for value in item.values()]

for path in paths_to_check:
    print(f"{path}: exists={path.exists()}")


In [ ]:
price_manifest_15m = prices_by_timeframe[TIMEFRAME_15M]
signal_manifests_15m = {
    indicator_id: yaml.safe_load(paths["manifest"].read_text())
    for indicator_id, paths in EXTRA_SIGNAL_PATHS_15M.items()
}
hit_times_manifest = yaml.safe_load(HIT_TIMES_MANIFEST_PATH.read_text())

print("15m price bars:", price_manifest_15m["coverage"]["bar_count"])
for indicator_id, doc in signal_manifests_15m.items():
    print(
        indicator_id,
        "rows_count=", doc["rows_count"],
        "shape=", tuple(doc["signals"]["shape"]),
        "signal_defaults=", doc["grid"].get("signals_v1_params_defaults", {}),
    )
print("hit_times tables:", hit_times_manifest["tables"])
print("tp grid path:", hit_times_manifest["tp_values"])
print("sl grid path:", hit_times_manifest["sl_values"])


In [ ]:
price_open_time_15m = np.load(PRICE_OPEN_TIME_15M_PATH, mmap_mode="r")
price_close_time_15m = np.load(PRICE_CLOSE_TIME_15M_PATH, mmap_mode="r")
price_ohlcv_15m = np.load(PRICE_OHLCV_15M_PATH, mmap_mode="r")

signal_matrices_15m = {
    indicator_id: np.load(paths["signals"], mmap_mode="r")
    for indicator_id, paths in EXTRA_SIGNAL_PATHS_15M.items()
}

tp_values_1m = np.load(TP_VALUES_PATH, mmap_mode="r")
sl_values_1m = np.load(SL_VALUES_PATH, mmap_mode="r")
long_tp_1m = np.load(LONG_TP_PATH, mmap_mode="r")
short_tp_1m = np.load(SHORT_TP_PATH, mmap_mode="r")
long_sl_1m = np.load(LONG_SL_PATH, mmap_mode="r")
short_sl_1m = np.load(SHORT_SL_PATH, mmap_mode="r")

print("price_open_time_15m:", price_open_time_15m.shape, price_open_time_15m.dtype)
print("price_close_time_15m:", price_close_time_15m.shape, price_close_time_15m.dtype)
print("price_ohlcv_15m:", price_ohlcv_15m.shape, price_ohlcv_15m.dtype)
for indicator_id, matrix in signal_matrices_15m.items():
    print(indicator_id, matrix.shape, matrix.dtype)
print("tp_values_1m:", tp_values_1m.shape, tp_values_1m.dtype)
print("sl_values_1m:", sl_values_1m.shape, sl_values_1m.dtype)
print("long_tp_1m:", long_tp_1m.shape, long_tp_1m.dtype)
print("short_tp_1m:", short_tp_1m.shape, short_tp_1m.dtype)
print("long_sl_1m:", long_sl_1m.shape, long_sl_1m.dtype)
print("short_sl_1m:", short_sl_1m.shape, short_sl_1m.dtype)


In [ ]:
open_time_index_15m = price_open_time_15m.astype("datetime64[ms]")
close_time_index_15m = price_close_time_15m.astype("datetime64[ms]")
time_mask_15m = (price_open_time_15m >= RUN_TIME_RANGE_START_MS) & (price_open_time_15m <= RUN_TIME_RANGE_END_MS)

price_fields_15m = {
    "open": np.asarray(price_ohlcv_15m[:, 0], dtype=np.float32),
    "high": np.asarray(price_ohlcv_15m[:, 1], dtype=np.float32),
    "low": np.asarray(price_ohlcv_15m[:, 2], dtype=np.float32),
    "close": np.asarray(price_ohlcv_15m[:, 3], dtype=np.float32),
    "volume": np.asarray(price_ohlcv_15m[:, 4], dtype=np.float32),
}
price_fields_15m["hlc3"] = (price_fields_15m["high"] + price_fields_15m["low"] + price_fields_15m["close"]) / 3.0
price_fields_15m["ohlc4"] = (price_fields_15m["open"] + price_fields_15m["high"] + price_fields_15m["low"] + price_fields_15m["close"]) / 4.0

print("15m bars inside run time range:", int(time_mask_15m.sum()))
print("first 15m bar:", np.datetime_as_string(open_time_index_15m[time_mask_15m][0], timezone="UTC"))
print("last 15m bar:", np.datetime_as_string(open_time_index_15m[time_mask_15m][-1], timezone="UTC"))
print("derived price fields:", tuple(price_fields_15m.keys()))


In [ ]:
def build_source_blocks(matrix, *, sources, param_values):
    block = len(param_values)
    return {
        source: matrix[idx * block:(idx + 1) * block]
        for idx, source in enumerate(sources)
    }

signal_source_blocks_15m = {
    indicator_id: build_source_blocks(
        signal_matrices_15m[indicator_id],
        sources=meta["sources"],
        param_values=meta["param_values"],
    )
    for indicator_id, meta in EXTRA_INDICATORS_15M.items()
}

for indicator_id, source_map in signal_source_blocks_15m.items():
    print("indicator:", indicator_id)
    for source_name, block in source_map.items():
        print("  ", source_name, block.shape, block.dtype)


In [ ]:
sample_indices_15m = np.flatnonzero(time_mask_15m)[:5]
sample_prices_15m = [
    {
        "open_time": np.datetime_as_string(open_time_index_15m[idx], timezone="UTC"),
        "close_time": np.datetime_as_string(close_time_index_15m[idx], timezone="UTC"),
        "open": float(price_fields_15m["open"][idx]),
        "high": float(price_fields_15m["high"][idx]),
        "low": float(price_fields_15m["low"][idx]),
        "close": float(price_fields_15m["close"][idx]),
        "hlc3": float(price_fields_15m["hlc3"][idx]),
        "ohlc4": float(price_fields_15m["ohlc4"][idx]),
        "volume": float(price_fields_15m["volume"][idx]),
    }
    for idx in sample_indices_15m
]
sample_prices_15m


In [ ]:
signal_probe_15m = []
for idx in np.flatnonzero(time_mask_15m)[:8]:
    signal_probe_15m.append({
        "open_time": np.datetime_as_string(open_time_index_15m[idx], timezone="UTC"),
        "ma.sma.close.row_0": int(signal_source_blocks_15m["ma.sma"]["close"][0, idx]),
        "ma.sma.open.row_0": int(signal_source_blocks_15m["ma.sma"]["open"][0, idx]),
        "momentum.roc.low.row_0": int(signal_source_blocks_15m["momentum.roc"]["low"][0, idx]),
        "momentum.roc.ohlc4.row_0": int(signal_source_blocks_15m["momentum.roc"]["ohlc4"][0, idx]),
        "volatility.stddev.high.row_0": int(signal_source_blocks_15m["volatility.stddev"]["high"][0, idx]),
        "volatility.stddev.close.row_0": int(signal_source_blocks_15m["volatility.stddev"]["close"][0, idx]),
    })
signal_probe_15m


## Full 15m five-indicator experiment with TP/SL

This section extends the notebook to five indicators on `15m` for `BTCUSDT spot binance`:

- `ma.dema`
- `ma.ema`
- `ma.sma`
- `momentum.roc`
- `volatility.stddev`

Assumptions for this experimental engine:

- all available artifact source blocks are loaded (`close`, `hlc3`, `ohlc4`, `low`, `high`, `open`)
- timeframe is `15m`
- TP/SL grid is `1.0% .. 30.0%` with `0.5%` step
- precomputed artifact `hit_times` are still loaded for inspection, but the kernel below uses direct `1m` high/low scans because the current artifact table only covers `0.5% .. 3.0%`
- the full five-indicator cartesian space is astronomically large, so the engine below is designed for chunked row-pool search rather than pretending that a full exhaustive sweep is cheap


In [ ]:
import heapq
from math import prod

from numba import njit

COMMON_PRICE_SOURCES = ["close", "hlc3", "ohlc4", "low", "high", "open"]
FULL_INDICATORS_15M = {
    "ma.dema": {
        "family": "ma",
        "sources": COMMON_PRICE_SOURCES,
        "param_name": "window",
        "param_values": list(range(5, 201)),
    },
    "ma.ema": {
        "family": "ma",
        "sources": COMMON_PRICE_SOURCES,
        "param_name": "window",
        "param_values": list(range(5, 201)),
    },
    "ma.sma": {
        "family": "ma",
        "sources": COMMON_PRICE_SOURCES,
        "param_name": "window",
        "param_values": list(range(5, 201)),
    },
    "momentum.roc": {
        "family": "momentum",
        "sources": COMMON_PRICE_SOURCES,
        "param_name": "window",
        "param_values": [5, 7, 10, 14, 21, 28, 42, 63, 84, 126],
    },
    "volatility.stddev": {
        "family": "volatility",
        "sources": COMMON_PRICE_SOURCES,
        "param_name": "window",
        "param_values": [10, 14, 20, 28, 42, 56, 84, 126],
    },
}

FULL_SIGNAL_PATHS_15M = {
    indicator_id: {
        "manifest": ARTIFACT_ROOT / "signals" / TIMEFRAME_15M / indicator_id / "manifest.yaml",
        "signals": ARTIFACT_ROOT / "signals" / TIMEFRAME_15M / indicator_id / "signals.i8.npy",
    }
    for indicator_id in FULL_INDICATORS_15M
}

MAPPINGS_15M_DIR = ARTIFACT_ROOT / "mappings" / TIMEFRAME_15M
BAR_OPEN_1M_IDX_15M_PATH = MAPPINGS_15M_DIR / "bar_open_1m_idx.u32.npy"
BAR_CLOSE_1M_IDX_15M_PATH = MAPPINGS_15M_DIR / "bar_close_1m_idx.u32.npy"

PRICE_DIR_1M = ARTIFACT_ROOT / "prices" / "1m"
PRICE_OPEN_TIME_1M_PATH = PRICE_DIR_1M / "open_time.i64.npy"
PRICE_CLOSE_TIME_1M_PATH = PRICE_DIR_1M / "close_time.i64.npy"
PRICE_OHLCV_1M_PATH = PRICE_DIR_1M / "ohlcv.f32.npy"

FIVE_INDICATOR_PATHS = [
    value
    for item in FULL_SIGNAL_PATHS_15M.values()
    for value in item.values()
] + [
    BAR_OPEN_1M_IDX_15M_PATH,
    BAR_CLOSE_1M_IDX_15M_PATH,
    PRICE_OPEN_TIME_1M_PATH,
    PRICE_CLOSE_TIME_1M_PATH,
    PRICE_OHLCV_1M_PATH,
]

for path in FIVE_INDICATOR_PATHS:
    print(f"{path}: exists={path.exists()}")


In [ ]:
full_signal_manifests_15m = {
    indicator_id: yaml.safe_load(paths["manifest"].read_text())
    for indicator_id, paths in FULL_SIGNAL_PATHS_15M.items()
}

full_signal_matrices_15m = {
    indicator_id: np.load(paths["signals"], mmap_mode="r")
    for indicator_id, paths in FULL_SIGNAL_PATHS_15M.items()
}

bar_open_1m_idx_15m = np.load(BAR_OPEN_1M_IDX_15M_PATH, mmap_mode="r")
bar_close_1m_idx_15m = np.load(BAR_CLOSE_1M_IDX_15M_PATH, mmap_mode="r")
price_open_time_1m = np.load(PRICE_OPEN_TIME_1M_PATH, mmap_mode="r")
price_close_time_1m = np.load(PRICE_CLOSE_TIME_1M_PATH, mmap_mode="r")
price_ohlcv_1m = np.load(PRICE_OHLCV_1M_PATH, mmap_mode="r")

price_fields_1m = {
    "open": np.asarray(price_ohlcv_1m[:, 0], dtype=np.float32),
    "high": np.asarray(price_ohlcv_1m[:, 1], dtype=np.float32),
    "low": np.asarray(price_ohlcv_1m[:, 2], dtype=np.float32),
    "close": np.asarray(price_ohlcv_1m[:, 3], dtype=np.float32),
    "volume": np.asarray(price_ohlcv_1m[:, 4], dtype=np.float32),
}

tp_sl_grid = np.arange(0.01, 0.3001, 0.005, dtype=np.float32)

run_open_15m = np.asarray(price_fields_15m["open"][time_mask_15m], dtype=np.float32)
run_close_15m = np.asarray(price_fields_15m["close"][time_mask_15m], dtype=np.float32)
run_bar_open_1m_idx_15m = np.asarray(bar_open_1m_idx_15m[time_mask_15m], dtype=np.int64)
run_bar_close_1m_idx_15m = np.asarray(bar_close_1m_idx_15m[time_mask_15m], dtype=np.int64)

for indicator_id, matrix in full_signal_matrices_15m.items():
    print(indicator_id, matrix.shape, matrix.dtype)
print("run_open_15m", run_open_15m.shape, run_open_15m.dtype)
print("run_close_15m", run_close_15m.shape, run_close_15m.dtype)
print("run_bar_open_1m_idx_15m", run_bar_open_1m_idx_15m.shape, run_bar_open_1m_idx_15m.dtype)
print("run_bar_close_1m_idx_15m", run_bar_close_1m_idx_15m.shape, run_bar_close_1m_idx_15m.dtype)
print("tp_sl_grid", tp_sl_grid[:5], "...", tp_sl_grid[-5:], "count=", tp_sl_grid.size)
print("artifact hit_times tp values", np.asarray(tp_values_1m))
print("artifact hit_times sl values", np.asarray(sl_values_1m))


In [ ]:
def build_row_catalog(*, indicator_id, sources, param_name, param_values):
    """Build row metadata for one artifact-backed indicator matrix.

    Parameters:
        indicator_id: Indicator id matching the artifact directory name.
        sources: Ordered artifact source axis for the indicator.
        param_name: Human-readable parameter axis name.
        param_values: Ordered values for the parameter axis.

    Returns:
        A list of dictionaries, one per matrix row, with row id, source, and parameter value.

    Assumptions:
        The artifact row order is `source-major` and then `param_values` inside each source block.

    Errors:
        Does not raise on its own; caller is responsible for providing consistent metadata.

    Side effects:
        None.
    """
    rows = []
    block = len(param_values)
    for source_idx, source_name in enumerate(sources):
        base = source_idx * block
        for offset, param_value in enumerate(param_values):
            rows.append({
                "indicator_id": indicator_id,
                "row_id": base + offset,
                "source": source_name,
                param_name: int(param_value),
            })
    return rows


def row_ids_for_sources(*, indicator_id, source_names):
    """Return row ids for the requested sources of one indicator.

    Parameters:
        indicator_id: Indicator id present in `FULL_INDICATORS_15M`.
        source_names: Iterable of source names such as `close` or `ohlc4`.

    Returns:
        A contiguous `int32` array of row ids matching the requested source blocks.

    Assumptions:
        The indicator uses the common source-major artifact row layout.

    Errors:
        Raises `KeyError` if an unknown source name is requested.

    Side effects:
        None.
    """
    meta = FULL_INDICATORS_15M[indicator_id]
    param_count = len(meta["param_values"])
    source_to_index = {name: idx for idx, name in enumerate(meta["sources"])}
    row_ids = []
    for source_name in source_names:
        source_idx = source_to_index[source_name]
        start = source_idx * param_count
        row_ids.extend(range(start, start + param_count))
    return np.asarray(row_ids, dtype=np.int32)


five_indicator_source_blocks_15m = {
    indicator_id: build_source_blocks(
        full_signal_matrices_15m[indicator_id],
        sources=meta["sources"],
        param_values=meta["param_values"],
    )
    for indicator_id, meta in FULL_INDICATORS_15M.items()
}

five_indicator_row_catalogs_15m = {
    indicator_id: build_row_catalog(
        indicator_id=indicator_id,
        sources=meta["sources"],
        param_name=meta["param_name"],
        param_values=meta["param_values"],
    )
    for indicator_id, meta in FULL_INDICATORS_15M.items()
}

five_indicator_row_counts = {
    indicator_id: int(matrix.shape[0])
    for indicator_id, matrix in full_signal_matrices_15m.items()
}
full_variant_count_5 = prod(five_indicator_row_counts.values())
full_variant_count_with_tp_sl = full_variant_count_5 * int(tp_sl_grid.size) * int(tp_sl_grid.size)

print("five indicator row counts:", five_indicator_row_counts)
print("plain five-indicator combinations:", full_variant_count_5)
print("five-indicator combinations with tp/sl grid:", full_variant_count_with_tp_sl)
print("sample rows for ma.ema", five_indicator_row_catalogs_15m["ma.ema"][:3])
print("sample rows for volatility.stddev", five_indicator_row_catalogs_15m["volatility.stddev"][:3])


In [ ]:
@njit
def scan_tp_sl_exit_1m(position, entry_price, tp_pct, sl_pct, high_1m, low_1m, start_idx, end_idx):
    """Scan one intrabar 1m interval for TP/SL exit.

    Parameters:
        position: Current position sign (`1` for long, `-1` for short).
        entry_price: Entry price for the open position.
        tp_pct: Take-profit percentage as decimal, for example `0.01` for `1%`.
        sl_pct: Stop-loss percentage as decimal.
        high_1m: Full 1m high series.
        low_1m: Full 1m low series.
        start_idx: Inclusive 1m bar index where the scan starts.
        end_idx: Inclusive 1m bar index where the scan ends.

    Returns:
        `(hit, exit_price)` where `hit` is `1` if an exit happened and `0` otherwise.

    Assumptions:
        If both TP and SL are touched within the same 1m candle, the function applies a conservative adverse-first rule.

    Errors:
        Does not raise; invalid ranges simply produce `no hit`.

    Side effects:
        None.
    """
    if start_idx < 0 or end_idx < start_idx:
        return 0, entry_price

    if position == 1:
        tp_price = entry_price * (1.0 + tp_pct)
        sl_price = entry_price * (1.0 - sl_pct)
        for idx in range(start_idx, end_idx + 1):
            low_value = low_1m[idx]
            high_value = high_1m[idx]
            if low_value <= sl_price and high_value >= tp_price:
                return 1, sl_price
            if low_value <= sl_price:
                return 1, sl_price
            if high_value >= tp_price:
                return 1, tp_price
        return 0, entry_price

    tp_price = entry_price * (1.0 - tp_pct)
    sl_price = entry_price * (1.0 + sl_pct)
    for idx in range(start_idx, end_idx + 1):
        low_value = low_1m[idx]
        high_value = high_1m[idx]
        if high_value >= sl_price and low_value <= tp_price:
            return 1, sl_price
        if high_value >= sl_price:
            return 1, sl_price
        if low_value <= tp_price:
            return 1, tp_price
    return 0, entry_price


@njit
def backtest_five_indicator_combo(
    dema_row,
    ema_row,
    sma_row,
    roc_row,
    stddev_row,
    open_15m,
    close_15m,
    high_1m,
    low_1m,
    bar_open_1m_idx,
    bar_close_1m_idx,
    tp_pct,
    sl_pct,
):
    """Backtest one five-indicator combination on 15m signals with 1m TP/SL replay.

    Parameters:
        dema_row: One `ma.dema` signal row aligned to the 15m bar axis.
        ema_row: One `ma.ema` signal row aligned to the 15m bar axis.
        sma_row: One `ma.sma` signal row aligned to the 15m bar axis.
        roc_row: One `momentum.roc` signal row aligned to the 15m bar axis.
        stddev_row: One `volatility.stddev` signal row aligned to the 15m bar axis.
        open_15m: 15m open prices for the selected run range.
        close_15m: 15m close prices for the selected run range.
        high_1m: 1m high prices for the full artifact range.
        low_1m: 1m low prices for the full artifact range.
        bar_open_1m_idx: Mapping from each 15m bar to the inclusive open 1m index.
        bar_close_1m_idx: Mapping from each 15m bar to the inclusive close 1m index.
        tp_pct: Take-profit percentage as decimal.
        sl_pct: Stop-loss percentage as decimal.

    Returns:
        Total compounded return in percent for this one combination.

    Assumptions:
        - a position is opened at the next 15m open after a consensus signal appears on the previous bar
        - a position is reversed at the next 15m open when the full five-indicator consensus flips
        - neutral consensus keeps the current position unchanged
        - final open position is liquidated at the last 15m close

    Errors:
        Does not raise; the caller must ensure aligned arrays and valid mappings.

    Side effects:
        None.
    """
    equity = 1.0
    position = 0
    entry_price = 0.0
    bars = open_15m.shape[0]

    for bar_idx in range(1, bars):
        signal_idx = bar_idx - 1
        long_ready = (
            dema_row[signal_idx] == 1
            and ema_row[signal_idx] == 1
            and sma_row[signal_idx] == 1
            and roc_row[signal_idx] == 1
            and stddev_row[signal_idx] == 1
        )
        short_ready = (
            dema_row[signal_idx] == -1
            and ema_row[signal_idx] == -1
            and sma_row[signal_idx] == -1
            and roc_row[signal_idx] == -1
            and stddev_row[signal_idx] == -1
        )

        desired_position = position
        if long_ready:
            desired_position = 1
        elif short_ready:
            desired_position = -1

        current_open = open_15m[bar_idx]

        if position == 0 and desired_position != 0:
            position = desired_position
            entry_price = current_open
        elif position != 0 and desired_position == -position:
            if position == 1:
                equity *= current_open / entry_price
            else:
                equity *= 2.0 - (current_open / entry_price)
            position = desired_position
            entry_price = current_open

        if position != 0:
            hit, exit_price = scan_tp_sl_exit_1m(
                position,
                entry_price,
                tp_pct,
                sl_pct,
                high_1m,
                low_1m,
                bar_open_1m_idx[bar_idx],
                bar_close_1m_idx[bar_idx],
            )
            if hit == 1:
                if position == 1:
                    equity *= exit_price / entry_price
                else:
                    equity *= 2.0 - (exit_price / entry_price)
                position = 0
                entry_price = 0.0

    if position != 0:
        final_close = close_15m[-1]
        if position == 1:
            equity *= final_close / entry_price
        else:
            equity *= 2.0 - (final_close / entry_price)

    return (equity - 1.0) * 100.0


@njit
def evaluate_combo_over_tp_sl_grid(
    dema_row,
    ema_row,
    sma_row,
    roc_row,
    stddev_row,
    open_15m,
    close_15m,
    high_1m,
    low_1m,
    bar_open_1m_idx,
    bar_close_1m_idx,
    tp_grid,
    sl_grid,
):
    """Evaluate one five-indicator row combination on the full TP/SL grid.

    Parameters:
        Same signal and price arrays as `backtest_five_indicator_combo`, plus TP and SL grids.

    Returns:
        A `float64` matrix of shape `[len(tp_grid), len(sl_grid)]` with total return percent values.

    Assumptions:
        The caller uses this helper for bounded row pools rather than for the full impossible five-way search space.

    Errors:
        Does not raise on its own; invalid caller inputs will surface inside the nested kernel.

    Side effects:
        None.
    """
    out = np.empty((tp_grid.shape[0], sl_grid.shape[0]), dtype=np.float64)
    for tp_idx in range(tp_grid.shape[0]):
        for sl_idx in range(sl_grid.shape[0]):
            out[tp_idx, sl_idx] = backtest_five_indicator_combo(
                dema_row,
                ema_row,
                sma_row,
                roc_row,
                stddev_row,
                open_15m,
                close_15m,
                high_1m,
                low_1m,
                bar_open_1m_idx,
                bar_close_1m_idx,
                tp_grid[tp_idx],
                sl_grid[sl_idx],
            )
    return out


In [ ]:
def estimate_combo_count(row_pools, *, tp_grid, sl_grid):
    """Estimate the total number of evaluations for a bounded five-indicator search.

    Parameters:
        row_pools: Mapping of indicator id to candidate row ids.
        tp_grid: TP grid as decimal percentages.
        sl_grid: SL grid as decimal percentages.

    Returns:
        Integer total evaluation count equal to the cartesian product of row pools and TP/SL pairs.

    Assumptions:
        `row_pools` contains one entry for each of the five indicators.

    Errors:
        Raises `KeyError` if a required indicator pool is absent.

    Side effects:
        None.
    """
    indicator_order = ("ma.dema", "ma.ema", "ma.sma", "momentum.roc", "volatility.stddev")
    return int(prod(len(np.asarray(row_pools[indicator_id])) for indicator_id in indicator_order) * len(tp_grid) * len(sl_grid))


def search_topk_five_indicator_combinations(*, row_pools, tp_grid, sl_grid, top_k=100):
    """Search the best TP/SL-adjusted combinations over bounded row pools.

    Parameters:
        row_pools: Mapping of indicator id to candidate row ids.
        tp_grid: TP percentages as decimals.
        sl_grid: SL percentages as decimals.
        top_k: Number of best rows to retain in the result heap.

    Returns:
        A descending list of dictionaries with indicator rows, TP/SL values, and `total_return_pct`.

    Assumptions:
        This helper is intended for bounded experimental pools. The full five-indicator cartesian product is too large to run exhaustively.

    Errors:
        Raises `KeyError` if any required indicator pool is missing.

    Side effects:
        Compiles numba kernels on first invocation, which can make the first call slower.
    """
    indicator_order = ("ma.dema", "ma.ema", "ma.sma", "momentum.roc", "volatility.stddev")
    normalized_pools = {
        indicator_id: np.asarray(row_pools[indicator_id], dtype=np.int32)
        for indicator_id in indicator_order
    }

    heap = []

    for dema_row_id in normalized_pools["ma.dema"]:
        dema_row = np.asarray(full_signal_matrices_15m["ma.dema"][int(dema_row_id), time_mask_15m], dtype=np.int8)
        for ema_row_id in normalized_pools["ma.ema"]:
            ema_row = np.asarray(full_signal_matrices_15m["ma.ema"][int(ema_row_id), time_mask_15m], dtype=np.int8)
            for sma_row_id in normalized_pools["ma.sma"]:
                sma_row = np.asarray(full_signal_matrices_15m["ma.sma"][int(sma_row_id), time_mask_15m], dtype=np.int8)
                for roc_row_id in normalized_pools["momentum.roc"]:
                    roc_row = np.asarray(full_signal_matrices_15m["momentum.roc"][int(roc_row_id), time_mask_15m], dtype=np.int8)
                    for stddev_row_id in normalized_pools["volatility.stddev"]:
                        stddev_row = np.asarray(full_signal_matrices_15m["volatility.stddev"][int(stddev_row_id), time_mask_15m], dtype=np.int8)
                        grid_scores = evaluate_combo_over_tp_sl_grid(
                            dema_row,
                            ema_row,
                            sma_row,
                            roc_row,
                            stddev_row,
                            run_open_15m,
                            run_close_15m,
                            price_fields_1m["high"],
                            price_fields_1m["low"],
                            run_bar_open_1m_idx_15m,
                            run_bar_close_1m_idx_15m,
                            np.asarray(tp_grid, dtype=np.float32),
                            np.asarray(sl_grid, dtype=np.float32),
                        )
                        for tp_idx, tp_pct in enumerate(tp_grid):
                            for sl_idx, sl_pct in enumerate(sl_grid):
                                score = float(grid_scores[tp_idx, sl_idx])
                                item = (
                                    score,
                                    {
                                        "total_return_pct": score,
                                        "tp_pct": float(tp_pct * 100.0),
                                        "sl_pct": float(sl_pct * 100.0),
                                        "ma.dema_row_id": int(dema_row_id),
                                        "ma.ema_row_id": int(ema_row_id),
                                        "ma.sma_row_id": int(sma_row_id),
                                        "momentum.roc_row_id": int(roc_row_id),
                                        "volatility.stddev_row_id": int(stddev_row_id),
                                    },
                                )
                                if len(heap) < top_k:
                                    heapq.heappush(heap, item)
                                elif score > heap[0][0]:
                                    heapq.heapreplace(heap, item)

    return [item for _, item in sorted(heap, key=lambda pair: pair[0], reverse=True)]


all_rows_5 = {
    indicator_id: np.arange(matrix.shape[0], dtype=np.int32)
    for indicator_id, matrix in full_signal_matrices_15m.items()
}

bounded_demo_rows_5 = {
    "ma.dema": row_ids_for_sources(indicator_id="ma.dema", source_names=["close"])[:2],
    "ma.ema": row_ids_for_sources(indicator_id="ma.ema", source_names=["close"])[:2],
    "ma.sma": row_ids_for_sources(indicator_id="ma.sma", source_names=["close"])[:2],
    "momentum.roc": row_ids_for_sources(indicator_id="momentum.roc", source_names=["close"])[:2],
    "volatility.stddev": row_ids_for_sources(indicator_id="volatility.stddev", source_names=["close"])[:2],
}

print("full search evaluations", estimate_combo_count(all_rows_5, tp_grid=tp_sl_grid, sl_grid=tp_sl_grid))
print("bounded demo evaluations", estimate_combo_count(bounded_demo_rows_5, tp_grid=tp_sl_grid[:3], sl_grid=tp_sl_grid[:3]))
bounded_demo_rows_5
